# Bébé JEPA multisensoriel — version Kaggle

**Avant de lancer (une seule fois) :**
1. *Settings → Accelerator* : **GPU P100** (un seul GPU est utilisé ; « T4 ×2 » marche aussi mais n'en utilise qu'un).
2. *Settings → Internet* : **On** (pour `git clone`; le téléphone doit être vérifié sur le compte Kaggle).
3. *Add Input* : le Dataset Kaggle qui contient les checkpoints (ex. `phase0_rD17n_0e_190k.pt` pour les évaluations,
   et `phase0_rD17n.pt` — le fichier de **reprise** avec l'optimiseur — pour continuer un entraînement).

**Long calcul sans garder le navigateur ouvert :** *Save Version → Save & Run All (Commit)*. Tout ce qui est écrit dans
`/kaggle/working/jepa_runs` est gardé dans l'onglet *Output* de la version (≤ 20 Go).

Choisis la tâche dans la cellule 2 (`TACHE = ...`), puis *Run All*.


In [ ]:
# 1. Installation : code du projet (GitHub) + lejepa ; GPU
import os, glob, shutil, subprocess
REPO = "/tmp/jepa-physics-bench"                      # hors de /kaggle/working : le code n'encombre pas les sorties
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-q", "https://github.com/frpatry/jepa-physics-bench", REPO], check=True)
subprocess.run(["git", "-C", REPO, "pull", "-q"], check=True)
print(subprocess.run(["git", "-C", REPO, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
!pip -q install git+https://github.com/rbalestr-lab/lejepa && echo LEJEPA OK
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
D = "/kaggle/working/jepa_runs"; os.makedirs(D, exist_ok=True)
CKPTS = sorted(glob.glob("/kaggle/input/**/*.pt", recursive=True))
print("checkpoints trouvés :", *CKPTS, sep="\n  ")


In [ ]:
# 2. Choix de la tâche
TACHE = "diag"          # "diag" (6 s : score image-but de scènes) | "gifs" (~15 min) | "train" (entraînement, des heures)
EVAL_CKPT = next((c for c in CKPTS if c.endswith("_0e_190k.pt")), CKPTS[0] if CKPTS else None)   # instantané à évaluer
RESUME = next((c for c in CKPTS if os.path.basename(c) == "phase0_rD17n.pt"), None)               # reprise (avec l'optimiseur)
NOM = "rK"              # nom du nouvel entraînement : fichiers phase0_<NOM>.pt et phase0_<NOM>_0e_<k>k.pt dans jepa_runs
TOTAL = 202000          # pas visés (D17n s'est arrêté à 190k ; ~1 000–1 500 pas / h sur P100 : ajuster à la session de ≤ 12 h)
print(TACHE, "| éval :", EVAL_CKPT, "| reprise :", RESUME)


In [ ]:
# 3a. DIAG du score image-but (av_goalcost_diag.py) — quelques secondes
if TACHE == "diag":
    !cd {REPO} && python -u av_goalcost_diag.py --ckpt {EVAL_CKPT} 2>&1 | tee {D}/run_goalcost_diag.log


In [ ]:
# 3b. GIF du bébé sans aide (Push-T avec / sans proposition, objet) — but en pointillés, couverture en direct
if TACHE == "gifs":
    G = f"{D}/gifs"; os.makedirs(G, exist_ok=True)
    P = f"--ckpt {EVAL_CKPT} --diag 0 --pols= --plan_hs 1 --film_dir {G} --gif_frames 300"
    !cd {REPO} && python -u av_plan0.py {P} --task pusht --slide 1 --plan_steps 300 --goal_img 1 --stop_success 1 --episodes 4 --propose 1 --gif 4 --gif_tag prop > {D}/run_gif_pusht_prop.log 2>&1
    !cd {REPO} && python -u av_plan0.py {P} --task pusht --slide 1 --plan_steps 300 --goal_img 1 --stop_success 1 --episodes 4 --propose 0 --gif 4 --gif_tag mpc > {D}/run_gif_pusht_mpc.log 2>&1
    !cd {REPO} && python -u av_plan0.py {P} --task objet --slide 1 --plan_steps 48 --goal_img 1 --episodes 4 --propose 1 --gif 4 --gif_tag img > {D}/run_gif_objet.log 2>&1
    !grep -hE "MPC|couverture|INTUITION" {D}/run_gif_*.log; ls {G}


In [ ]:
# 3c. ENTRAÎNEMENT (recette D17n) — reprise depuis le fichier de reprise ; la précision passe en fp16 automatiquement sur P100/T4
if TACHE == "train":
    ck = f"{D}/phase0_{NOM}.pt"
    if not os.path.exists(ck): shutil.copy(RESUME, ck)     # les entrées Kaggle sont en lecture seule : on travaille sur une copie
    RECETTE = ("--touch_masks 1 --contact_w 4 --p_obj 0.8 --mob_size 0.16 --calm 2 --outside 1 --parent 0.4 --inv_clip 1 --ctx_w 0.5 "
               "--ctx_on enc --causal 1 --fut_share 0.7 --blind 1 --curious 3 --echo 0.5 --damp 0.5 --eyes_share 0.15 --replay 0.1 "
               "--motion_w 1.0 --shape_phys 1 --roll_w 1.0 --roll_k 3 --roll_obj 4 --goal_w 0.3 --goal_k 8 --goal_obj 4 --goal_lr 1.0 --goal_nohand 1")
    # (pour un NOUVEAU cycle : ajouter --lr_restart <pas de départ> --lr_peak 0.15, et changer la recette — UNE chose à la fois)
    !cd {REPO} && python -u av_phase0.py --ckpt {ck} --total {TOTAL} {RECETTE} >> {D}/run_phase0_{NOM}.log 2>&1
    !tail -5 {D}/run_phase0_{NOM}.log; ls -la {D}


In [ ]:
# 4. État (à relancer pendant une session interactive)
!date; ls -la {D}; for f in {D}/*.log; do echo "## $$f"; tail -3 "$$f"; done
